# 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
3. We store those pieces in a searchable database (a "vector store").
4. When a user asks a question, we **search** the database for the most relevant pieces, and hand them to an AI model to **generate** a final answer.

That's it. No magic — just search + a language model.

**What we use in this notebook:**
- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> Before running: make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file with `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

## Step 1 — Import everything we need

We import all the tools upfront so it's clear what's being used and where it comes from.

In [2]:
import os 
from dotenv import load_dotenv
from pathlib import Path

# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings



C:\Users\HP\AppData\Local\Temp\ipykernel_26236\1544617503.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
load_dotenv()

True

In [4]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")

ENV VAR LOADED


LOADING OUR DATA

In [5]:
BASE_DIR = Path.cwd().parent
print(f"BASE_DIR: {BASE_DIR}")

BASE_DIR: d:\Workshop\practice-rag


In [6]:
DATA_FILE_PATH = BASE_DIR / "data" / "basic-rag" / "hr_policy.txt"
print(f"DATA_FILE_PATH: {DATA_FILE_PATH}")

DATA_FILE_PATH: d:\Workshop\practice-rag\data\basic-rag\hr_policy.txt


#### DATA INGESTION 

In [7]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: d:\Workshop\practice-rag\data\basic-rag\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2597

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


#### LANGCHAIN DOCUMENT 

Langchain processes everything in form of documents 


DOCUMENTS : 

PAGE CONTENT -- the actual data 

METADATA  - extra information about the data 

In [8]:
len(documents)

1

In [9]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [10]:
print(documents[0].metadata)

{'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}


In [11]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2597


In [12]:
print(f"Document ID: {documents[0].id}")

Document ID: None


SPLITTING OUR DATA

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 9


In [14]:
# Print preview of first 2 chunks
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Chunk ID: {chunk.id}")
    print(f"Chunk Metadata: {chunk.metadata}")
    print(f"Chunk Content Preview: {chunk.page_content}")


--- Chunk 1 ---
Chunk ID: None
Chunk Metadata: {'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}
Chunk Content Preview: COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

--- Chunk 2 ---
Chunk ID: None
Chunk Metadata: {'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}
Chunk Content Preview: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Chunk 3 ---
Chunk ID: None
Chunk Metadata: {'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}
Chunk Content Preview: 2. WORK FROM HOME POLICY
Employees may work

NOW EACH SPILTED CHUNK IS A DOCUMENT - page content and metadata

In [15]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}


In [16]:
print(chunks[5])

page_content='5. REIMBURSEMENT POLICY
Employees can claim reimbursement for approved business expenses such as travel,
client meals, and internet bills used for official work.
All reimbursement claims must be submitted with valid bills within 30 days of the expense.
Claims are processed within 10 working days after approval from the reporting manager.' metadata={'source': 'd:\\Workshop\\practice-rag\\data\\basic-rag\\hr_policy.txt'}


In [17]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


In [18]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [19]:
print(chunks[6].page_content)

6. CODE OF CONDUCT
Employees are expected to maintain professionalism and respect in the workplace.
Harassment, discrimination, or any form of workplace misconduct will not be tolerated
and may result in disciplinary action, including termination.
All employees must complete an annual Code of Conduct training.


EMBEDD OUR DATA

In [20]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)

EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


### STORE DATA IN VECTOR DB

In [21]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


WE NEVER STORED IT 

In [22]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()


Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



#### TOOL

In [23]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

### DATA RETRIVAL

LLM 

In [24]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [25]:
tr = llm.invoke("Hey what is the leave policy")

In [26]:
print(tr.content)

Sure! While the exact details can vary from one organization (or country) to another, most leave policies cover a few common categories:

| Leave Type | Typical Eligibility | Typical Amount (per year) | Key Points |
|------------|--------------------|---------------------------|------------|
| **Vacation / Paid Time Off (PTO)** | Full‑time employees (often after a probation period) | 10‑30 days, increasing with tenure | Often accrued monthly; may roll over a limited amount or be “use‑or‑lose.” |
| **Sick Leave** | All employees (sometimes separate from PTO) | 5‑10 days (sometimes unlimited) | Usually requires a doctor’s note after a certain number of days. |
| **Personal/Administrative Leave** | All employees | 1‑5 days | For errands, appointments, or other non‑vacation needs. |
| **Parental / Maternity / Paternity Leave** | Employees with a newborn or adopted child | 6‑24 weeks (paid or unpaid) | Governed by local law (e.g., FMLA in the U.S., EU parental leave directives). |
| **Berea

AI AGENT

3 -- 

LLM - BRAIN 

TOOL - SUPER POWER 

MEMORY - no memory 

In [27]:
from langchain.agents import create_agent      

In [28]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Corp. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [29]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [30]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)


In [31]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='a78d136c-4ee6-4443-b7f5-5b6a025dec17'),
  AIMessage(content='I’m the friendly HR assistant here at **Acme\u202fCorp**. How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". As HR assistant, we should answer that we work for Acme Corp. No need to search policy. It\'s not about policy. We can answer directly.'}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 208, 'total_tokens': 284, 'completion_time': 0.15624748, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.008819922, 'prompt_tokens_details': None, 'queue_time': 0.403887579, 'total_time': 0.165067402}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b3ed-f977-711

SYSTEM MESSAGE - HR ASSISNT

HUMAN MESSAGE - TELL ME ABOUT POLICIES 

AI MESSAGE  - HEY THESE ARE THEPLOICES 


In [32]:
response["messages"][-2].content

'tell me which org you work for'

In [33]:
print(response["messages"][-1].content)

I’m the friendly HR assistant here at **Acme Corp**. How can I help you today?


In [34]:
for i in range(len(response["messages"])):
    print(f"{response['messages'][i].type} message: {response['messages'][i].content}")

human message: tell me which org you work for
ai message: I’m the friendly HR assistant here at **Acme Corp**. How can I help you today?


In [35]:
question = "How many sick leaves employees get"

response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": question
            }
        ]
    }
)

In [36]:
response

{'messages': [HumanMessage(content='How many sick leaves employees get', additional_kwargs={}, response_metadata={}, id='423c6721-697c-4b67-97ea-74dc4cd2fdfd'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer: "How many sick leaves employees get". Must use search_hr_policy tool to look up facts. So we call search_hr_policy with question.', 'tool_calls': [{'id': 'fc_8ae6f769-c2a5-4948-84ca-76dabbc67a7f', 'function': {'arguments': '{"question":"How many sick leaves employees get"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 207, 'total_tokens': 274, 'completion_time': 0.143157152, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.088007133, 'prompt_tokens_details': None, 'queue_time': 0.350126569, 'total_time': 0.231164285}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': '

In [44]:
for i in range(len(response["messages"])):
    print(f"{response['messages'][i].type} message: {response['messages'][i].content}")

human message: How many sick leaves employees get
ai message: 
tool message: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Perfo

In [45]:
response["messages"]

[HumanMessage(content='How many sick leaves employees get', additional_kwargs={}, response_metadata={}, id='423c6721-697c-4b67-97ea-74dc4cd2fdfd'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer: "How many sick leaves employees get". Must use search_hr_policy tool to look up facts. So we call search_hr_policy with question.', 'tool_calls': [{'id': 'fc_8ae6f769-c2a5-4948-84ca-76dabbc67a7f', 'function': {'arguments': '{"question":"How many sick leaves employees get"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 207, 'total_tokens': 274, 'completion_time': 0.143157152, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.088007133, 'prompt_tokens_details': None, 'queue_time': 0.350126569, 'total_time': 0.231164285}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', '

In [46]:
response["messages"][-1]

AIMessage(content='Employees are entitled to **10 paid sick days per year**.', additional_kwargs={'reasoning_content': 'We have the policy text. The question: "How many sick leaves employees get". The policy says "employees get 10 paid sick days per year". So answer that.'}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 449, 'total_tokens': 507, 'completion_time': 0.123405828, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.028593101, 'prompt_tokens_details': None, 'queue_time': 0.402516205, 'total_time': 0.151998929}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_9241e9962b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0b3ee-1119-7ca2-93da-0aa344734fa0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 449, 'output_tokens': 58, 'total_tokens': 507, 'output_token_details': {'reasoning': 36}})

In [47]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

tool_calls = response["messages"][-1].tool_calls

In [42]:
made_tool_calls = len(tool_calls) > 0

In [43]:
print(tool_calls)

[]
